# Assignment 2 — MLP Classification on the Iris Dataset

**Platform:** Google Colab &nbsp;|&nbsp; **Suggested runtime:** CPU  
**How to use:** Run the cells from top to bottom. Change the small experiment
constants when more training time is available.

This workbook is written as a compact college assignment: it explains the
problem, implements the method, evaluates the result, and records the main
observations.


## Problem and method

Build a Multilayer Perceptron (MLP) that predicts one of three Iris
species from four flower measurements. Dense hidden layers learn
non-linear feature combinations; softmax returns class probabilities.

**Evaluation:** test accuracy, per-class precision/recall/F1, and a
confusion matrix.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data.astype("float32"), iris.target.astype("int32"),
    test_size=0.20, random_state=SEED, stratify=iris.target
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dropout(0.10),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax"),
], name="iris_mlp")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

history = model.fit(
    X_train, y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=16,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=12, restore_best_weights=True
    )],
    verbose=0,
)
print("Epochs completed:", len(history.history["loss"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set(title="Loss", xlabel="Epoch"); axes[0].legend()
axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set(title="Accuracy", xlabel="Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
probabilities = model.predict(X_test, verbose=0)
predictions = probabilities.argmax(axis=1)

print(f"Test accuracy: {test_accuracy:.3f}")
print(classification_report(y_test, predictions, target_names=iris.target_names))

cm = confusion_matrix(y_test, predictions)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("MLP confusion matrix")
plt.show()


## Interpretation

Diagonal confusion-matrix cells are correct predictions; off-diagonal
cells are mistakes. Setosa is normally easiest to separate, while
versicolor and virginica may overlap. A good report states the achieved
accuracy and identifies which pair of classes was confused most often.


## Conclusion

The experiment above provides a complete training and evaluation workflow. The
printed metrics and plots are the result for the current run and should be used
to identify the strongest behaviour, the main limitation, and one justified
improvement. Exact values may vary slightly because neural-network training is
stochastic.
